# 12 Joins Padron — Joins y Operaciones de Conjunto

## Dataset
Padrón Electoral (`PADRON_COMPLETO.csv`, ~3.5 M filas) y `DISTELEC.csv` en Unity Catalog Volumes.

In [0]:
# En Databricks Free Edition el objeto `spark` ya existe — no se necesita SparkSession.builder
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql.functions import col, desc, countDistinct, current_date, broadcast
from time import time

BASE = "/Volumes/big_data_2026/spark_examples/spark_data"

Listamos los archivos disponibles en el Volumen para confirmar que el Padrón y DISTELEC existen antes de cargarlos.

In [0]:
display(dbutils.fs.ls(BASE))

Cargamos el Padrón (`padronDf`) y el catálogo de distritos `DISTELEC` (`distelecDf`) con schema explícito, derivado del `Leame.txt` del Tribunal Supremo de Elecciones. `DISTELEC` es una tabla pequeña (miles de filas) frente a los ~3.5 M del Padrón — clave para el experimento de broadcast más adelante.

In [0]:
padron_schema = StructType([
    StructField("CEDULA",     IntegerType(), True),
    StructField("CODELEC",    IntegerType(), True),
    StructField("RELLENO",    StringType(),  True),
    StructField("FECHACADUC", DateType(),    True),
    StructField("JUNTA",      IntegerType(), True),
    StructField("NOMBRE",     StringType(),  True),
    StructField("1_APELLIDO", StringType(),  True),
    StructField("2_APELLIDO", StringType(),  True),
])
padronDf = (spark.read.option("header", True).option("dateFormat", "yyyyMMdd")
            .schema(padron_schema).csv(f"{BASE}/PADRON_COMPLETO.csv"))

# Catálogo de distritos (clave CODELEC -> nombres). Tabla pequeña: ideal para broadcast.
distelec_schema = StructType([
    StructField("CODELEC",   IntegerType(), True),
    StructField("PROVINCIA", StringType(),  True),
    StructField("CANTON",    StringType(),  True),
    StructField("DISTRITO",  StringType(),  True),
])
distelecDf = (spark.read.option("header", True)
              .schema(distelec_schema).csv(f"{BASE}/DISTELEC.csv"))

print("Padrón:", padronDf.count(), "filas |", "DISTELEC:", distelecDf.count(), "filas")

## Inner join: cada votante con su provincia, cantón y distrito

¿El inner join puede dejar fuera votantes cuyo CODELEC no esté en DISTELEC?

In [0]:
votantes_inner = padronDf.join(distelecDf, on="CODELEC", how="inner")
print("Filas Padrón:", padronDf.count(), "| Filas tras inner join:", votantes_inner.count())
votantes_inner.select("CEDULA", "NOMBRE", "1_APELLIDO", "PROVINCIA", "CANTON", "DISTRITO").show(5)

## Left join: identificar CODELEC sin coincidencia en DISTELEC

¿Qué columnas quedarán en `null` si un CODELEC no tiene match?

In [0]:
votantes_left = padronDf.join(distelecDf, on="CODELEC", how="left")
sin_match = votantes_left.filter(col("PROVINCIA").isNull())
print("Votantes totales:", votantes_left.count())
print("Votantes con CODELEC sin coincidencia en DISTELEC:", sin_match.count())
sin_match.select("CEDULA", "CODELEC", "PROVINCIA", "CANTON", "DISTRITO").show(5)

## Broadcast join: medir el efecto de evitar el shuffle de la tabla grande

DISTELEC es pequeña y el padrón enorme. ¿Será más rápido el broadcast o el join normal?

In [0]:
# Join normal: el optimizador decide la estrategia (puede requerir shuffle/Exchange de ambas tablas)
t0 = time()
join_normal = padronDf.join(distelecDf, on="CODELEC", how="inner")
join_normal.write.format("noop").mode("overwrite").save()
print("Join normal →", round(time() - t0, 2), "s")

# Broadcast join: la tabla pequeña (DISTELEC) se envía completa a cada ejecutor, evitando el shuffle del Padrón
t0 = time()
join_broadcast = padronDf.join(broadcast(distelecDf), on="CODELEC", how="inner")
join_broadcast.write.format("noop").mode("overwrite").save()
print("Broadcast join →", round(time() - t0, 2), "s")

Comparamos los planes físicos: el join normal puede mostrar `SortMergeJoin`/`Exchange` (shuffle de ambas tablas), mientras que el broadcast muestra `BroadcastHashJoin` (la tabla chica viaja completa a cada ejecutor, sin shuffle del Padrón).

In [0]:
print("=== Plan físico: join normal ===")
join_normal.explain()

print("\n=== Plan físico: broadcast join ===")
join_broadcast.explain()

## Cross join controlado: producto cartesiano sobre subconjuntos pequeños

Si cruzo 3 filas con 3 filas, ¿cuántas filas salen?

In [0]:
# Subconjuntos pequeños: 3 distritos x 3 juntas (sin riesgo de explotar memoria)
distritos_chicos = distelecDf.select("DISTRITO").distinct().limit(3)
juntas_chicas = padronDf.select("JUNTA").distinct().limit(3)

producto = distritos_chicos.crossJoin(juntas_chicas)
print("Distritos:", distritos_chicos.count(), "| Juntas:", juntas_chicas.count(), "| Cross join:", producto.count())
producto.show()

## Operaciones de conjunto: union y unionByName

Simulamos dos "provincias" filtrando el Padrón en dos DataFrames disjuntos y los apilamos de nuevo con `union`.

Si apilo dos DataFrames con `union`, ¿se eliminan los duplicados automáticamente?

In [0]:
# Simulamos dos "provincias" disjuntas filtrando por CODELEC y las apilamos
votantes_con_provincia = padronDf.join(distelecDf.select("CODELEC", "PROVINCIA"), on="CODELEC", how="inner")
provincia_a = votantes_con_provincia.filter(col("PROVINCIA") == "SAN JOSE")
provincia_b = votantes_con_provincia.filter(col("PROVINCIA") == "ALAJUELA")

apilado = provincia_a.union(provincia_b)
print("Filas A:", provincia_a.count(), "| Filas B:", provincia_b.count(), "| Filas tras union:", apilado.count())
assert apilado.count() == provincia_a.count() + provincia_b.count(), "El total debería coincidir: union NO elimina duplicados"

# unionByName: igual que union pero alinea columnas por nombre, no por posición
apilado_by_name = provincia_a.unionByName(provincia_b)
print("Filas tras unionByName:", apilado_by_name.count())

## distinct, intersect y subtract sobre subconjuntos pequeños

¿Cuántas provincias quedarán si combino la lista de cantones de San José con la de Alajuela y elimino duplicados?

In [0]:
cantones_sj = distelecDf.filter(col("PROVINCIA") == "SAN JOSE").select("CANTON").distinct()
cantones_al = distelecDf.filter(col("PROVINCIA") == "ALAJUELA").select("CANTON").distinct()

# distinct: ya aplicado arriba con .distinct() para quedarnos con valores únicos
print("Cantones SJ:", cantones_sj.count(), "| Cantones Alajuela:", cantones_al.count())

# intersect: nombres de cantón que coinciden entre ambas provincias (si los hay)
comunes = cantones_sj.intersect(cantones_al)
print("Cantones con nombre repetido entre SJ y Alajuela:", comunes.count())
comunes.show()

# subtract: cantones de SJ que NO aparecen en la lista de Alajuela
solo_sj = cantones_sj.subtract(cantones_al)
print("Cantones exclusivos de SJ:", solo_sj.count())

## Agregación tras el join: votantes por provincia/cantón y cédulas caducadas

Con el resultado del inner join, contamos votantes únicos (`countDistinct("CEDULA")`) por `PROVINCIA` y `CANTON`, y calculamos la proporción de cédulas con `FECHACADUC` ya vencida.

In [0]:
# Votantes únicos por provincia y cantón
votantes_por_zona = (votantes_inner
                      .groupBy("PROVINCIA", "CANTON")
                      .agg(countDistinct("CEDULA").alias("votantes"))
                      .orderBy(desc("votantes")))
votantes_por_zona.show(10)

# Proporción de cédulas con fecha de caducidad ya vencida
total = votantes_inner.count()
caducadas = votantes_inner.filter(col("FECHACADUC") < current_date()).count()
print(f"Cédulas caducadas: {caducadas:,} de {total:,} ({100 * caducadas / total:.2f}%)")